In [291]:
# ----------------------------------------------------------------------------
# 1. IMPORTS
# ----------------------------------------------------------------------------
import pandas as pd
import numpy as np
from datetime import datetime

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.impute import SimpleImputer

# Modelos
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

# Feature Engineering
from sklearn.cluster import KMeans

# Hyper Params Tuning
from sklearn.model_selection import GridSearchCV

In [292]:
# ----------------------------------------------------------------------------
# 2. CARREGAR DADOS
# ----------------------------------------------------------------------------

dfTrain = pd.read_csv('training_data.csv', encoding='latin1')
dfTest = pd.read_csv('test_data.csv', encoding='latin1')

In [293]:
dfTrain.shape

(6812, 14)

In [294]:
dfTest.shape

(1500, 13)

In [295]:
dfTrain.head()

,city_name,record_date,AVERAGE_SPEED_DIFF,AVERAGE_FREE_FLOW_SPEED,AVERAGE_TIME_DIFF,AVERAGE_FREE_FLOW_TIME,LUMINOSITY,AVERAGE_TEMPERATURE,AVERAGE_ATMOSP_PRESSURE,AVERAGE_HUMIDITY,AVERAGE_WIND_SPEED,AVERAGE_CLOUDINESS,AVERAGE_PRECIPITATION,AVERAGE_RAIN
0,Porto,2019-08-29 07:00:00,Medium,41.5,11.5,71.4,LIGHT,15.0,1019.0,100.0,3.0,NaN,0.0,NaN
1,Porto,2018-08-10 14:00:00,High,41.7,48.3,87.4,LIGHT,21.0,1021.0,53.0,5.0,céu claro,0.0,NaN
2,Porto,2019-09-01 16:00:00,High,38.6,38.4,85.2,LIGHT,26.0,1014.0,61.0,4.0,NaN,0.0,NaN
3,Porto,2019-02-26 11:00:00,High,37.4,61.0,94.1,LIGHT,18.0,1025.0,48.0,4.0,céu claro,0.0,NaN
4,Porto,2019-06-06 12:00:00,Medium,41.6,50.4,77.0,LIGHT,15.0,1008.0,82.0,10.0,NaN,0.0,NaN


In [296]:
dfTest.head()

,city_name,record_date,AVERAGE_FREE_FLOW_SPEED,AVERAGE_TIME_DIFF,AVERAGE_FREE_FLOW_TIME,LUMINOSITY,AVERAGE_TEMPERATURE,AVERAGE_ATMOSP_PRESSURE,AVERAGE_HUMIDITY,AVERAGE_WIND_SPEED,AVERAGE_CLOUDINESS,AVERAGE_PRECIPITATION,AVERAGE_RAIN
0,Porto,2019-02-13 23:00:00,39.2,0.0,91.0,DARK,8.0,1026.0,71.0,1.0,céu claro,0.0,NaN
1,Porto,2018-11-28 20:00:00,42.5,12.2,76.8,DARK,11.0,1020.0,93.0,4.0,nuvens dispersas,0.0,NaN
2,Porto,2018-08-14 05:00:00,45.9,0.0,86.3,DARK,14.0,1017.0,93.0,0.0,NaN,0.0,NaN
3,Porto,2019-07-06 17:00:00,33.2,51.7,89.9,LIGHT,22.0,1016.0,77.0,4.0,céu pouco nublado,0.0,NaN
4,Porto,2018-10-15 06:00:00,44.0,3.5,85.5,DARK,12.0,1004.0,100.0,9.0,NaN,0.0,chuva fraca


In [297]:
dfTrain.columns

Index(['city_name', 'record_date', 'AVERAGE_SPEED_DIFF',
       'AVERAGE_FREE_FLOW_SPEED', 'AVERAGE_TIME_DIFF',
       'AVERAGE_FREE_FLOW_TIME', 'LUMINOSITY', 'AVERAGE_TEMPERATURE',
       'AVERAGE_ATMOSP_PRESSURE', 'AVERAGE_HUMIDITY', 'AVERAGE_WIND_SPEED',
       'AVERAGE_CLOUDINESS', 'AVERAGE_PRECIPITATION', 'AVERAGE_RAIN'],
      dtype='object')

In [298]:
dfTest.columns

Index(['city_name', 'record_date', 'AVERAGE_FREE_FLOW_SPEED',
       'AVERAGE_TIME_DIFF', 'AVERAGE_FREE_FLOW_TIME', 'LUMINOSITY',
       'AVERAGE_TEMPERATURE', 'AVERAGE_ATMOSP_PRESSURE', 'AVERAGE_HUMIDITY',
       'AVERAGE_WIND_SPEED', 'AVERAGE_CLOUDINESS', 'AVERAGE_PRECIPITATION',
       'AVERAGE_RAIN'],
      dtype='object')

In [299]:
dfTrain = dfTrain.drop(columns=['city_name'])
dfTest = dfTest.drop(columns=['city_name'])

In [300]:
dfTrain['AVERAGE_SPEED_DIFF'].unique()

array(['Medium', 'High', nan, 'Low', 'Very_High'], dtype=object)

In [301]:
print(dfTrain['AVERAGE_SPEED_DIFF'].isna().sum())

2200


In [302]:
## O QUE FAÇO COM TANTOS Nan
# Preenche os valores NaN com a string 'Missing'
dfTrain['AVERAGE_SPEED_DIFF'] = dfTrain['AVERAGE_SPEED_DIFF'].fillna('Missing')

In [303]:
def extract_datetime_features(df):
    """Extrai features temporais da coluna record_date"""
    df = df.copy()
    
    # Converter para datetime
    df['record_date'] = pd.to_datetime(df['record_date'])
    
    # Extrair componentes
    df['month'] = df['record_date'].dt.month
    df['hour'] = df['record_date'].dt.hour
    df['day_of_week'] = df['record_date'].dt.dayofweek
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    
    return df

In [304]:
dfTrain = extract_datetime_features(dfTrain)
dfTest = extract_datetime_features(dfTest)
dfTrain = dfTrain.drop(columns=['record_date'])
dfTest = dfTest.drop(columns=['record_date'])

In [305]:
dfTrain['hour'].value_counts()

hour
22    301
20    296
3     295
6     289
8     289
1     287
18    287
2     286
21    285
13    285
17    284
19    284
7     283
12    282
9     282
14    282
16    281
15    280
10    280
0     278
23    276
5     275
11    273
4     272
Name: count, dtype: int64

In [306]:
dfTest['hour'].value_counts()

hour
4     80
5     74
0     70
11    70
23    69
15    68
2     65
17    64
16    64
10    63
1     63
12    63
19    63
9     62
14    62
18    60
21    59
13    59
6     58
7     57
3     57
8     55
20    49
22    46
Name: count, dtype: int64

In [307]:
dfTrain['is_rush_hour'] = ((dfTrain['hour'] >= 7) & (dfTrain['hour'] <= 10) | 
                          (dfTrain['hour'] >= 16) & (dfTrain['hour'] <= 21)).astype(int)
dfTest['is_rush_hour'] = ((dfTest['hour'] >= 7) & (dfTest['hour'] <= 10) | 
                          (dfTest['hour'] >= 16) & (dfTest['hour'] <= 21)).astype(int)

In [308]:
dfTrain['LUMINOSITY'].value_counts()

LUMINOSITY
LIGHT        3293
DARK         3253
LOW_LIGHT     266
Name: count, dtype: int64

In [309]:
# Converter LUMINOSITY para 0/1 (DARK=0, LIGHT=1)
def convert_luminosity(df):
    df = df.copy()
    df['LUMINOSITY'] = df['LUMINOSITY'].map({'DARK': 0, 'LIGHT': 1, 'LOW_LIGHT': 2})
    return df

In [310]:
dfTrain = convert_luminosity(dfTrain)
dfTest = convert_luminosity(dfTest)
dfTrain = dfTrain.drop(columns=['LUMINOSITY'])
dfTest = dfTest.drop(columns=['LUMINOSITY'])

In [311]:
dfTrain['hour'].value_counts()

hour
22    301
20    296
3     295
6     289
8     289
1     287
18    287
2     286
21    285
13    285
17    284
19    284
7     283
12    282
9     282
14    282
16    281
15    280
10    280
0     278
23    276
5     275
11    273
4     272
Name: count, dtype: int64

In [312]:
dfTrain['AVERAGE_CLOUDINESS'].unique()

array([nan, 'céu claro', 'nuvens dispersas', 'céu pouco nublado',
       'céu limpo', 'algumas nuvens', 'nuvens quebrados', 'tempo nublado',
       'nuvens quebradas', 'nublado'], dtype=object)

In [313]:
dfTest['AVERAGE_CLOUDINESS'].unique()

array(['céu claro', 'nuvens dispersas', nan, 'céu pouco nublado',
       'nuvens quebradas', 'algumas nuvens', 'nuvens quebrados',
       'nublado', 'céu limpo', 'tempo nublado'], dtype=object)

In [314]:
dfTrain['AVERAGE_RAIN'].unique()

array([nan, 'chuva fraca', 'chuva', 'chuva leve', 'chuvisco fraco',
       'chuva moderada', 'trovoada com chuva leve', 'aguaceiros',
       'aguaceiros fracos', 'chuva de intensidade pesada',
       'trovoada com chuva', 'chuva de intensidade pesado', 'chuva forte',
       'chuvisco e chuva fraca'], dtype=object)

In [315]:
dfTest['AVERAGE_RAIN'].unique()

array([nan, 'chuva fraca', 'chuva moderada', 'chuva', 'aguaceiros fracos',
       'chuva leve', 'aguaceiros', 'trovoada com chuva', 'chuvisco fraco',
       'trovoada com chuva leve'], dtype=object)

In [320]:
dfTrain.isna().sum()

AVERAGE_SPEED_DIFF         0
AVERAGE_FREE_FLOW_SPEED    0
AVERAGE_TIME_DIFF          0
AVERAGE_FREE_FLOW_TIME     0
AVERAGE_TEMPERATURE        0
AVERAGE_ATMOSP_PRESSURE    0
AVERAGE_HUMIDITY           0
AVERAGE_WIND_SPEED         0
AVERAGE_CLOUDINESS         0
AVERAGE_PRECIPITATION      0
AVERAGE_RAIN               0
month                      0
hour                       0
day_of_week                0
is_weekend                 0
is_rush_hour               0
dtype: int64

In [321]:
dfTest.isna().sum()

AVERAGE_FREE_FLOW_SPEED    0
AVERAGE_TIME_DIFF          0
AVERAGE_FREE_FLOW_TIME     0
AVERAGE_TEMPERATURE        0
AVERAGE_ATMOSP_PRESSURE    0
AVERAGE_HUMIDITY           0
AVERAGE_WIND_SPEED         0
AVERAGE_CLOUDINESS         0
AVERAGE_PRECIPITATION      0
AVERAGE_RAIN               0
month                      0
hour                       0
day_of_week                0
is_weekend                 0
is_rush_hour               0
dtype: int64

In [318]:
dfTrain = dfTrain.fillna('Missing')

In [319]:
dfTest = dfTest.fillna('Missing')